In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torchvision import datasets, models
from torch.utils.data import DataLoader

# ===============================
# 📌 Hiperparámetros
# ===============================
BATCH_SIZE = 32
LR = 0.001
EPOCHS = 10
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ===============================
# 📌 Transformaciones de imágenes
# ===============================
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# ===============================
# 📌 Cargar Dataset
# ===============================
data_dir = "dataset/dataset"
train_dataset = datasets.ImageFolder(root=f"{data_dir}/train", transform=transform)
val_dataset = datasets.ImageFolder(root=f"{data_dir}/test", transform=transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [12]:
train_dataset.samples

[('dataset/dataset/train/freshapples/Screen Shot 2018-06-08 at 4.59.36 PM.png',
  0),
 ('dataset/dataset/train/freshapples/Screen Shot 2018-06-08 at 4.59.49 PM.png',
  0),
 ('dataset/dataset/train/freshapples/Screen Shot 2018-06-08 at 4.59.57 PM.png',
  0),
 ('dataset/dataset/train/freshapples/Screen Shot 2018-06-08 at 5.00.03 PM.png',
  0),
 ('dataset/dataset/train/freshapples/Screen Shot 2018-06-08 at 5.00.12 PM.png',
  0),
 ('dataset/dataset/train/freshapples/Screen Shot 2018-06-08 at 5.00.18 PM.png',
  0),
 ('dataset/dataset/train/freshapples/Screen Shot 2018-06-08 at 5.00.26 PM.png',
  0),
 ('dataset/dataset/train/freshapples/Screen Shot 2018-06-08 at 5.00.35 PM.png',
  0),
 ('dataset/dataset/train/freshapples/Screen Shot 2018-06-08 at 5.00.43 PM.png',
  0),
 ('dataset/dataset/train/freshapples/Screen Shot 2018-06-08 at 5.00.50 PM.png',
  0),
 ('dataset/dataset/train/freshapples/Screen Shot 2018-06-08 at 5.01.01 PM.png',
  0),
 ('dataset/dataset/train/freshapples/Screen Shot 2018-

In [24]:
for label,numero in train_dataset.samples:
    print(label)
    print(numero)

dataset/dataset/train/freshapples/Screen Shot 2018-06-08 at 4.59.36 PM.png
0
dataset/dataset/train/freshapples/Screen Shot 2018-06-08 at 4.59.49 PM.png
0
dataset/dataset/train/freshapples/Screen Shot 2018-06-08 at 4.59.57 PM.png
0
dataset/dataset/train/freshapples/Screen Shot 2018-06-08 at 5.00.03 PM.png
0
dataset/dataset/train/freshapples/Screen Shot 2018-06-08 at 5.00.12 PM.png
0
dataset/dataset/train/freshapples/Screen Shot 2018-06-08 at 5.00.18 PM.png
0
dataset/dataset/train/freshapples/Screen Shot 2018-06-08 at 5.00.26 PM.png
0
dataset/dataset/train/freshapples/Screen Shot 2018-06-08 at 5.00.35 PM.png
0
dataset/dataset/train/freshapples/Screen Shot 2018-06-08 at 5.00.43 PM.png
0
dataset/dataset/train/freshapples/Screen Shot 2018-06-08 at 5.00.50 PM.png
0
dataset/dataset/train/freshapples/Screen Shot 2018-06-08 at 5.01.01 PM.png
0
dataset/dataset/train/freshapples/Screen Shot 2018-06-08 at 5.01.08 PM.png
0
dataset/dataset/train/freshapples/Screen Shot 2018-06-08 at 5.01.29 PM.png
0

In [2]:


# Obtener cantidad de clases
num_fruit_classes = len(set([label.split("_")[0] for _, label in train_dataset.samples]))  # Tipos de frutas
num_freshness_classes = len(set([label.split("_")[1] for _, label in train_dataset.samples]))  # Frescura

# ===============================
# 📌 Modelo EfficientNet con doble salida
# ===============================
class FruitClassifier(nn.Module):
    def __init__(self, num_fruit_classes, num_freshness_classes):
        super(FruitClassifier, self).__init__()
        self.backbone = models.efficientnet_b0(weights="IMAGENET1K_V1")
        self.backbone.classifier = nn.Identity()  # Quitar capa final

        # Salida 1: Tipo de fruta
        self.fc_fruit = nn.Sequential(
            nn.Linear(1280, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_fruit_classes)
        )

        # Salida 2: Frescura
        self.fc_freshness = nn.Sequential(
            nn.Linear(1280, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_freshness_classes)
        )

    def forward(self, x):
        features = self.backbone(x)
        fruit_pred = self.fc_fruit(features)
        freshness_pred = self.fc_freshness(features)
        return fruit_pred, freshness_pred

# ===============================
# 📌 Inicializar modelo, optimizador y función de pérdida
# ===============================
model = FruitClassifier(num_fruit_classes, num_freshness_classes).to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss()

# ===============================
# 📌 Función de Entrenamiento
# ===============================
def train(model, loader, optimizer, criterion):
    model.train()
    total_loss, total_correct_fruit, total_correct_fresh = 0, 0, 0
    total_samples = 0

    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()
        fruit_preds, fresh_preds = model(images)

        # Separar etiquetas
        fruit_labels = labels // num_freshness_classes  # Tipo de fruta
        fresh_labels = labels % num_freshness_classes  # Estado de frescura

        # Calcular pérdidas
        loss_fruit = criterion(fruit_preds, fruit_labels)
        loss_fresh = criterion(fresh_preds, fresh_labels)
        loss = loss_fruit + loss_fresh  # Optimización conjunta

        loss.backward()
        optimizer.step()

        # Métricas
        total_loss += loss.item()
        total_correct_fruit += (fruit_preds.argmax(1) == fruit_labels).sum().item()
        total_correct_fresh += (fresh_preds.argmax(1) == fresh_labels).sum().item()
        total_samples += labels.size(0)

    acc_fruit = total_correct_fruit / total_samples
    acc_fresh = total_correct_fresh / total_samples
    return total_loss / len(loader), acc_fruit, acc_fresh

# ===============================
# 📌 Función de Evaluación
# ===============================
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, total_correct_fruit, total_correct_fresh = 0, 0, 0
    total_samples = 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            fruit_preds, fresh_preds = model(images)

            # Separar etiquetas
            fruit_labels = labels // num_freshness_classes
            fresh_labels = labels % num_freshness_classes

            # Calcular pérdidas
            loss_fruit = criterion(fruit_preds, fruit_labels)
            loss_fresh = criterion(fresh_preds, fresh_labels)
            loss = loss_fruit + loss_fresh

            total_loss += loss.item()
            total_correct_fruit += (fruit_preds.argmax(1) == fruit_labels).sum().item()
            total_correct_fresh += (fresh_preds.argmax(1) == fresh_labels).sum().item()
            total_samples += labels.size(0)

    acc_fruit = total_correct_fruit / total_samples
    acc_fresh = total_correct_fresh / total_samples
    return total_loss / len(loader), acc_fruit, acc_fresh

# ===============================
# 📌 Entrenamiento del modelo
# ===============================
for epoch in range(EPOCHS):
    train_loss, train_acc_fruit, train_acc_fresh = train(model, train_loader, optimizer, criterion)
    val_loss, val_acc_fruit, val_acc_fresh = evaluate(model, val_loader, criterion)

    print(f"📢 Epoch {epoch+1}/{EPOCHS} | Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")
    print(f"🍏 Fruit Acc: {train_acc_fruit:.4f} (Train) | {val_acc_fruit:.4f} (Val)")
    print(f"🌿 Freshness Acc: {train_acc_fresh:.4f} (Train) | {val_acc_fresh:.4f} (Val)")


AttributeError: 'int' object has no attribute 'split'